# 03 - Combined Inference (MobileNet + MLP)


In [ ]:
import os
import cv2
import json
import numpy as np
import tensorflow as tf
import mediapipe as mp
from collections import deque


## Configuration & Model Loading


In [ ]:
IS_KAGGLE = os.path.exists('/kaggle/input')
MOBILENET_PATH = 'mobilenet_arabic_final.h5'
MLP_PATH = 'arsl_mediapipe_mlp_model_final.h5'
CLASSES_JSON = 'arabic_classes.json'

ALPHA_MLP = 0.6
ALPHA_MOBILE = 0.4
CONTROL_LABELS = ['space', 'del', 'nothing']

# 1. Load Classes
if os.path.exists(CLASSES_JSON):
    with open(CLASSES_JSON, 'r', encoding='utf-8') as f:
        classes = json.load(f)
else:
    print(f"WARNING: {CLASSES_JSON} not found. Fallback to extracting from dataset if exists.")
    classes = [] # Need robust logic here if you strictly rely on it

# 2. Load Models
print("Loading models...")
mobilenet_m = tf.keras.models.load_model(MOBILENET_PATH)
mlp_m = tf.keras.models.load_model(MLP_PATH)

# Critical checks
if mobilenet_m.output_shape[-1] != len(classes):
    raise ValueError(f"MobileNet outputs {mobilenet_m.output_shape[-1]} but json has {len(classes)} classes.")
if mlp_m.output_shape[-1] != len(classes):
    raise ValueError(f"MLP outputs {mlp_m.output_shape[-1]} but json has {len(classes)} classes.")

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)


## Inference Helper Functions


In [ ]:
def extract_landmarks(frame, results):
    if results.multi_hand_landmarks:
        lm = results.multi_hand_landmarks[0]
        return np.array([[l.x, l.y, l.z] for l in lm.landmark]).flatten()
    return np.zeros(63)

from PIL import ImageFont, ImageDraw, Image
def put_arabic_text(img, text, position=(50,50), font_size=32):
    # Handling arabic presentation (requires arabic_reshaper & bidi if available)
    # Basic fallback implementation
    try:
        import arabic_reshaper
        from bidi.algorithm import get_display
        text = get_display(arabic_reshaper.reshape(text))
    except ImportError:
        pass # Optional requirements for nice rendering
        
    img_pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil)
    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except IOError:
        font = ImageFont.load_default()
    draw.text(position, text, font=font, fill=(0, 255, 0, 0))
    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)


## Live Webcam / Static Image Testing


In [ ]:
if IS_KAGGLE:
    print("Kaggle detected. Providing static image test.")
    # Implement static inference here
    pass
else:
    print("Local detected. Launching webcam.")
    cap = cv2.VideoCapture(0)
    history = deque(maxlen=5)
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        # 1. MediaPipe
        results = hands.process(rgb)
        keypoints = extract_landmarks(frame, results)
        
        # Draw landmarks
        if results.multi_hand_landmarks:
            for hl in results.multi_hand_landmarks:
                mp.solutions.drawing_utils.draw_landmarks(frame, hl, mp_hands.HAND_CONNECTIONS)
                
        # 2. Predict MLP
        p_mlp = mlp_m.predict(np.expand_dims(keypoints, axis=0), verbose=0)[0]
        
        # 3. Predict MobileNet
        img_resized = cv2.resize(rgb, (128, 128))
        img_input = tf.keras.applications.mobilenet_v2.preprocess_input(img_resized)
        p_mobile = mobilenet_m.predict(np.expand_dims(img_input, axis=0), verbose=0)[0]
        
        # 4. Fusion
        if np.sum(keypoints) == 0:
            # If no hand detected, ignore MLP
            fused = p_mobile
        else:
            fused = p_mlp * ALPHA_MLP + p_mobile * ALPHA_MOBILE
            
        pred_idx = np.argmax(fused)
        history.append(pred_idx)
        
        # Temporal smoothing (majority vote)
        most_common = max(set(history), key=history.count)
        pred_class = classes[most_common]
        conf = fused[most_common]
        
        if pred_class in CONTROL_LABELS:
            pred_class = f"[ACTION: {pred_class}]"
            
        frame = put_arabic_text(frame, f"{pred_class} ({conf:.2f})", position=(30, 50))
        cv2.imshow('Sign Language Recognition', frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
    cap.release()
    cv2.destroyAllWindows()
